# THIGNS TO DO:
* Try to predict:
    - Using other blocks seperately.
    - Integrate more blocks.
    - Add features of between block changes, like change in average automaticity etc.

- Maygbe show how good it is with only 1 PC/few PCS.
- Do similarity/ditance of features + clustering
- RNN/LSTM...

- Remember my new findings about automaticity... (also note that i tested a model with only that and it does not predicti better than the PCA more elaborated model in most cases)

- MAYBE PC across all blocks and see what happens with PC1 for example along time... ...,

### Fututre maybe
* Maybe in the future: in the context of counterfactual training group/duration/
    - Maybe also try to extrapulate.. (not sure about this one.) -> now done with more HYPOTETICAL training see my notes in this part in case I want to do it with more training.


## Super NICE
- I checked also with removing bad cu=onsumption test. I think results are even better.
(block_data = block_data[~block_data['sub'].isin(POTENTIAL_EXCLUSIONS['CT_non_manip_minus_manip_lt_6_DF']['sub'].unique())].reset_index(drop=True))


# IMPORTANT
* CHECK OUT RESULTS IN: VERY IMPORTANT ABOUT AUTOMATICITY AND RELATED MEASURES


# NOTES:
* With specific PCs I can get much better results, but this is like double dipping, not good way to do it withoout it (I tried many things). But wiht the best PCs the model is much better!
* If we collect more data, we can test it the other data set I guess.

Instead of group, blocks of training... Actually it is the same after normalization when there are only two groups. So it does not really helps.
But we can still simulate someone with more training or less training (more than -1 and less then +1 before normalization).

- Tried PLS-DA before SVC, did not really work well (at least when each time I run it only on the training data and applied it on the test data along woith the SVC prediction attempt)

# Preparing stuff

# GET EVERYTHING READY

In [11]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib import ticker
import plotly.graph_objects as go

import numpy as np
import pandas as pd
import seaborn as sns
import os
import math


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import learning_curve

from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, roc_auc_score, auc, brier_score_loss, pairwise_distances, silhouette_samples, precision_score, recall_score, silhouette_score

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier # Multi layer perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay, partial_dependence
from sklearn.utils import shuffle
from sklearn.calibration import calibration_curve
from sklearn.tree import plot_tree
from sklearn.cross_decomposition import PLSRegression

import shap
from scipy.stats import binomtest, binom_test, ttest_ind, mannwhitneyu
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import chi2_contingency
from scipy.cluster.hierarchy import fcluster
from scipy.cluster.hierarchy import dendrogram

from joblib import Parallel, delayed
import pickle

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)



In [12]:
# READ the data:
main_data_df = pd.read_pickle('parsed_data/main_data_df_SAMPLE2.pkl')
CT_data = pd.read_pickle('parsed_data/CT_data_SAMPLE2.pkl')
gambles = pd.read_pickle('parsed_data/gambles_SAMPLE2.pkl')
demographic_data = pd.read_csv('parsed_data/demographic_data_STUDY2.csv')

main_data_df = main_data_df.reset_index(drop=True)
CT_data = CT_data.reset_index(drop=True)
gambles = gambles.reset_index(drop=True)
demographic_data = demographic_data.reset_index(drop=True)

# Helper functions for IPI calculations:
# -----------------------------
def extractCorrectKeyPresses(seq, presses):
    trial_key_presses = presses.copy()
    # remove key presses woth key_rt = None:
    trial_key_presses = [
        x for x in trial_key_presses if x['key_rt'] is not None]
    correct_presses = []
    min_loc_to_look = 0  # used to verify the order of the keys.
    if trial_key_presses:
        for i in range(len(seq)):
            trial_key_presses = trial_key_presses[min_loc_to_look:]
            if seq[i] in [x['key_pressed'] for x in trial_key_presses]:
                loc = [x['key_pressed']
                       for x in trial_key_presses].index(seq[i])
                correct_presses.append(trial_key_presses[loc])
                min_loc_to_look = loc + 1
    return correct_presses


def calculateIPIs(key_presses):
    ipis = np.nan
    if len(key_presses) == 3:
        IPI1 = key_presses[1]['key_rt'] - key_presses[0]['key_rt']
        IPI2 = key_presses[2]['key_rt'] - key_presses[1]['key_rt']
        ipis = [IPI1, IPI2]
    return ipis


def calc_IPI_Consistency(input_data, seperate_by_block=False):
    # cerate a copy of the data
    data = input_data.copy()
    for i in range(1, len(data)):
        row1 = data.iloc[i-1]
        row2 = data.iloc[i]
        isSameBlock = row1['block'] == row2['block']
        if row1['sub'] == row2['sub'] and row1['stimType'] == row2['stimType'] and \
                isinstance(row1['inter_press_intervals'], list) and isinstance(row2['inter_press_intervals'], list) and \
                row1['inter_press_intervals'] and row2['inter_press_intervals']:
            if not seperate_by_block or isSameBlock:
                # display(row1['inter_press_intervals'])
                # display(row2['inter_press_intervals'])
                IPI_1_abs_diff = abs(
                    row2['inter_press_intervals'][0] - row1['inter_press_intervals'][0])
                IPI_2_abs_diff = abs(
                    row2['inter_press_intervals'][1] - row1['inter_press_intervals'][1])
                # calculate the sum of the absolute difference between the IPIs:
                IPI_abs_diff_sum = IPI_1_abs_diff + IPI_2_abs_diff

                # add it to the dataframe:
                data.loc[i, 'IPI_abs_diff_sum'] = IPI_abs_diff_sum
    return data


# Create the IPI data:
# -----------------------------
IPI_consistency_data = main_data_df[main_data_df['blockType'] != 'gambles_only'].copy()

# create a new column based on SRO_keyPressSummary that includes only the correct sequence pressing (according to stim_seq)
IPI_consistency_data.loc[:, 'SRO_keyPressSummary_correct'] = IPI_consistency_data.apply(lambda row: extractCorrectKeyPresses(row['stim_seq'], row['SRO_keyPressSummary']), axis=1)
IPI_consistency_data.loc[:, 'inter_press_intervals'] = IPI_consistency_data.apply(lambda row: calculateIPIs(row['SRO_keyPressSummary_correct']), axis=1)

# Create a new dataframes with the IPIs (with and without block seperation)
sorted_IPI_consistency_data = IPI_consistency_data.sort_values(by=['sub', 'stimType', 'block', 'trial'])
sorted_IPI_consistency_data = sorted_IPI_consistency_data.reset_index(drop=True)

# add absolute stimulus trial number:
sorted_IPI_consistency_data.loc[:, 'stim_abs_trial'] = sorted_IPI_consistency_data.groupby(['sub', 'stimType']).cumcount() + 1

IPI_consistency_data_by_trial = calc_IPI_Consistency(sorted_IPI_consistency_data, seperate_by_block=False)
IPI_consistency_data_w_block_sep = calc_IPI_Consistency(sorted_IPI_consistency_data, seperate_by_block=True)

# add time befroe/after (deval) to main_data_df:
# -----------------------------------------------
main_data_df['time'] = np.nan
main_data_df.loc[main_data_df['phase'] == 'pre_test', 'time'] = 'before'
main_data_df.loc[main_data_df['phase'] == 'test', 'time'] = 'after'
main_data_df.loc[main_data_df['phase'] == 'reacquisition', 'time'] = 'after'
main_data_df.loc[:, ['sub', 'phase', 'time']].tail(100)

main_data_df = main_data_df.reset_index(drop=True)


/var/folders/h_/4yn2bj6118x8bmpj49gssb440000gn/T/ipykernel_27937/3930289685.py:87: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'before' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  main_data_df.loc[main_data_df['phase'] == 'pre_test', 'time'] = 'before'


# PRE-REGISTERED EXCLUSIONS

## Get exclusions

In [13]:
PRE_REG_EXCLUSIONS = {}

# get the number of n_pressing_gambleKeys_in_SRO as a table:
n_pressing_gambleKeys_in_SRO_summary = main_data_df.groupby(['group', 'sub', 'block'])['n_pressing_gambleKeys_in_SRO'].sum()
n_pressing_gambleKeys_in_SRO_summary = n_pressing_gambleKeys_in_SRO_summary.reset_index()
# make block values columns:
n_pressing_gambleKeys_in_SRO_summary_table = n_pressing_gambleKeys_in_SRO_summary.pivot(index='sub', columns='block', values='n_pressing_gambleKeys_in_SRO')
PRE_REG_EXCLUSIONS['has_a_block_with_more_than_50_gamble_key_presses_during_SRO'] = sorted(n_pressing_gambleKeys_in_SRO_summary.loc[n_pressing_gambleKeys_in_SRO_summary['n_pressing_gambleKeys_in_SRO'] > 50, 'sub'].unique().tolist())

# get the number of trials with at least one pressing of gamble keys:
trial_prop_w_gambleKeys_in_SRO = main_data_df.groupby(['group', 'sub', 'block'])['n_pressing_gambleKeys_in_SRO'].apply(lambda x: ((x > 0).sum() / len(x)).round(2)).reset_index()
# make block values columns:
trial_prop_w_gambleKeys_in_SRO_table = trial_prop_w_gambleKeys_in_SRO.pivot(index='sub', columns='block', values='n_pressing_gambleKeys_in_SRO')
PRE_REG_EXCLUSIONS['has_an_SRO_block_with_at_more_than_50_percent_trials_with_gamble_key_presses'] = sorted(trial_prop_w_gambleKeys_in_SRO.loc[trial_prop_w_gambleKeys_in_SRO['n_pressing_gambleKeys_in_SRO'] > 0.5, 'sub'].unique().tolist())
                   
# get the number of times 'choice_rt' is smaller than 200, per sub per block:
n_fast_choice_rt_summary = main_data_df.groupby(['group', 'sub', 'block'])['choice_rt'].apply(lambda x: ((x <= 200).sum() / len(x)).round(2))
n_fast_choice_rt_summary = n_fast_choice_rt_summary.reset_index()
# make block values columns:
n_fast_choice_rt_summary_table = n_fast_choice_rt_summary.pivot(index='sub', columns='block', values='choice_rt')
PRE_REG_EXCLUSIONS['has_amore_than_50_percent_gamble_trials_with_RT_less_than_200ms'] = sorted(n_fast_choice_rt_summary.loc[n_fast_choice_rt_summary['choice_rt'] > 0.5, 'sub'].unique().tolist())

# Get low SRO (Stimulus-Response-Outcome, i.e., operant learning part) engagement:
relev_SRO_data = main_data_df.loc[(main_data_df.stimType == 'planet_red') | (main_data_df.stimType == 'planet_blue'),:].copy().reset_index(drop=True)
# remove phase 'test' and 'reacquisition' (2nd test block):
relev_SRO_data = relev_SRO_data.loc[(relev_SRO_data.phase != 'test') & (relev_SRO_data.phase != 'reacquisition'),:].copy().reset_index(drop=True)
# Now for each sub, block, and group, calculate the proportion of SRO trials with sequenceCompleted:
SRO_response_rate = relev_SRO_data.groupby(['group', 'sub'])['sequenceCompleted'].apply(lambda x: (x.sum() / len(x)).round(2)).reset_index()
PRE_REG_EXCLUSIONS['has_less_than_60percent_Relevant_SRO_trials_with_sequenceCompleted'] = sorted(SRO_response_rate.loc[SRO_response_rate.sequenceCompleted < 0.6, 'sub'].unique().tolist())

# Get engagement toward the (never valued) YELLOW SRO trials:
relev_YELLOW_SRO_data = main_data_df.loc[main_data_df.stimType == 'planet_yellow',:].copy().reset_index(drop=True)
# Now for each sub, block, and group, calculate the proportion of SRO trials with sequenceCompleted:
SRO_YELLOW_response_rate = relev_YELLOW_SRO_data.groupby(['group', 'sub'])['sequenceCompleted'].apply(lambda x: (x.sum() / len(x)).round(2)).reset_index()
PRE_REG_EXCLUSIONS['has_more_than_40percent_Relevant_YELLOW_SRO_trials_with_sequenceCompleted'] = sorted(SRO_YELLOW_response_rate.loc[SRO_YELLOW_response_rate.sequenceCompleted > 0.4, 'sub'].unique().tolist())

# Get low response rate in gamble trials:
relev_gamble_data = main_data_df.loc[main_data_df.blockType != 'SRO_only',:].copy().reset_index(drop=True)
# Now for each sub, and group, calculate the proportion of SRO trials with sequenceCompleted:
Gamble_response_rate = relev_gamble_data.groupby(['group', 'sub'])['choice_rt'].apply(lambda x: (x.notnull().sum() / len(x)).round(2)).reset_index()
PRE_REG_EXCLUSIONS['has_less_than_60percent_Relevant_Gamble_trials_with_choice_rt'] = sorted(Gamble_response_rate.loc[Gamble_response_rate.choice_rt < 0.6, 'sub'].unique().tolist())


In [14]:
# now combine everything in PRE_REG_EXCLUSIONS to one list (ubique subjects) and sorted:
PRE_REG_EXCLUSIONS_LIST = sorted(set([item for sublist in PRE_REG_EXCLUSIONS.values() for item in sublist]))
print('Exlusions:', PRE_REG_EXCLUSIONS_LIST)
print('n_excluded:', len(PRE_REG_EXCLUSIONS_LIST))

# * Note on 5007: was timed out. from block 1 to 5 more than 2 hours had passed.
print('Adding 5007 to the exclusions list as he was *timed out* (took a long break in the middle of the task).')
PRE_REG_EXCLUSIONS_LIST = PRE_REG_EXCLUSIONS_LIST + [5007]
PRE_REG_EXCLUSIONS_LIST = sorted(PRE_REG_EXCLUSIONS_LIST)
print('Exlusions:', PRE_REG_EXCLUSIONS_LIST)
print('n_excluded:', len(PRE_REG_EXCLUSIONS_LIST))

# Use PRE_REG_EXCLUSIONS to print exclusion reasons:
print('\n\nExclusion reasons:')
for reason, subs in PRE_REG_EXCLUSIONS.items():
    if len(subs) > 0:
        print(f"{reason}: {', '.join(map(str, subs))} ({len(subs)} subjects)")

# Now per subject print all the reasons for exclusion:
print('\n\nExclusion reasons per subject:')
for sub in PRE_REG_EXCLUSIONS_LIST:
    reasons = []
    for reason, subs in PRE_REG_EXCLUSIONS.items():
        if sub in subs:
            reasons.append(reason)
    if len(reasons) > 0:
        print(f"Sub {sub}: {', '.join(reasons)} ({len(reasons)} reasons)")

Exlusions: [5006, 5008, 5017, 5035, 5043, 5060, 5067, 5069, 5085, 5087, 5092, 5093, 5094, 5096, 5103, 5129, 5134, 5138, 5140, 5149, 5161, 5164, 5166, 5169, 5181, 5188, 5192, 5203, 5208, 5216, 5217, 5218, 5238, 5240, 5245, 5251, 5252, 5262, 5268, 5272, 5275, 5280, 5294]
n_excluded: 43
Adding 5007 to the exclusions list as he was *timed out* (took a long break in the middle of the task).
Exlusions: [5006, 5007, 5008, 5017, 5035, 5043, 5060, 5067, 5069, 5085, 5087, 5092, 5093, 5094, 5096, 5103, 5129, 5134, 5138, 5140, 5149, 5161, 5164, 5166, 5169, 5181, 5188, 5192, 5203, 5208, 5216, 5217, 5218, 5238, 5240, 5245, 5251, 5252, 5262, 5268, 5272, 5275, 5280, 5294]
n_excluded: 44


Exclusion reasons:
has_a_block_with_more_than_50_gamble_key_presses_during_SRO: 5035, 5043, 5087, 5096, 5129, 5140, 5161, 5169, 5208, 5216, 5217, 5238, 5245, 5262, 5268, 5280, 5294 (17 subjects)
has_an_SRO_block_with_at_more_than_50_percent_trials_with_gamble_key_presses: 5006, 5008, 5035, 5043, 5085, 5087, 5092, 509

## Apply exclusion

In [15]:
print(PRE_REG_EXCLUSIONS_LIST)
# apply exclusions according to Exclusions variable:
main_data_df = main_data_df[~main_data_df['sub'].isin(PRE_REG_EXCLUSIONS_LIST)]
CT_data = CT_data[~CT_data['sub'].isin(PRE_REG_EXCLUSIONS_LIST)]
gambles = gambles[~gambles['sub'].isin(PRE_REG_EXCLUSIONS_LIST)]
IPI_consistency_data_by_trial = IPI_consistency_data_by_trial[~IPI_consistency_data_by_trial['sub'].isin(PRE_REG_EXCLUSIONS_LIST)]
demographic_data = demographic_data[~demographic_data['sub'].isin(PRE_REG_EXCLUSIONS_LIST)]

main_data_df = main_data_df.reset_index(drop=True)
CT_data = CT_data.reset_index(drop=True)
gambles = gambles.reset_index(drop=True)
IPI_consistency_data_by_trial = IPI_consistency_data_by_trial.reset_index(drop=True)
demographic_data = demographic_data.reset_index(drop=True)

[5006, 5007, 5008, 5017, 5035, 5043, 5060, 5067, 5069, 5085, 5087, 5092, 5093, 5094, 5096, 5103, 5129, 5134, 5138, 5140, 5149, 5161, 5164, 5166, 5169, 5181, 5188, 5192, 5203, 5208, 5216, 5217, 5218, 5238, 5240, 5245, 5251, 5252, 5262, 5268, 5272, 5275, 5280, 5294]


In [16]:
main_data_df['sub'].unique().shape

(258,)

## Remove phase 'shorter_time_test' (block 14)

In [17]:
main_data_df = main_data_df[main_data_df['phase'] != 'shorter_time_test'].reset_index(drop=True)
gambles = gambles[gambles['phase'] != 'shorter_time_test'].reset_index(drop=True)
IPI_consistency_data_by_trial = IPI_consistency_data_by_trial[IPI_consistency_data_by_trial['phase'] != 'shorter_time_test'].reset_index(drop=True)


## Calculate summary features, Construct habit expression data, and other stuff

In [18]:
# ---------------------------------------------------------
# Prepare data for analysis
# ---------------------------------------------------------

data = main_data_df.copy()

######### last_measured_automaticity (based on IPI_abs_diff_sum) #########
# integrate IPI_consistency_data_by_trial.IPI_abs_diff_sum with the data, get only IPI_abs_diff_sum from IPI_consistency_data_by_trial and add it to the data. Make sure that it is matched by sub, block, and trial:
IPI_abs_diff_sum = IPI_consistency_data_by_trial[['sub', 'block', 'trial', 'IPI_abs_diff_sum']]
data = pd.merge(data, IPI_abs_diff_sum, on=['sub', 'block', 'trial'], how='left')

columns_to_merge_gambles = [col for col in gambles.columns if col not in data.columns]
data = pd.merge(data, gambles[['sub', 'block', 'trial'] + columns_to_merge_gambles], on=['sub', 'block', 'trial'], how='left')


# ---------------------------------------------------------
# Calculate summary features per block:
# ---------------------------------------------------------
# initialize summary_features:
summary_features = pd.DataFrame()
for subj in data['sub'].unique():
    for block in data[data['sub']==subj]['block'].unique():
        sub_block_data = data[(data['sub']==subj) & (data['block']==block)]
        blockType = sub_block_data['blockType'].values[0]
        time = sub_block_data['time'].values[0]
        # get features:
        group = 0 # intermediate training (Study 2) will be coded as 0 (short and extensive training in Study 1 were coded as -1 and 1).        

        # gamble_stuff:
        choice_rt_mean = sub_block_data['choice_rt'].mean() if blockType != 'SRO_only' else np.nan
        choice_rt_std = sub_block_data['choice_rt'].std() if blockType != 'SRO_only' else np.nan
        chosen_location = sub_block_data['chosen_location'].value_counts().arrow_bottom if 'arrow_bottom' in sub_block_data['chosen_location'].value_counts() else 0 if blockType != 'SRO_only' else np.nan
        chosen_direction = sub_block_data['chosen_direction'].value_counts().left if 'left' in sub_block_data['chosen_direction'].value_counts() else 0 if blockType != 'SRO_only' else np.nan
        n_chosen_gambles = sub_block_data['chosen_gamble'].notna().sum() if blockType != 'SRO_only' else np.nan
        log_chosenEV_noChosenEV_ratio_mean = sub_block_data['log_chosenEV_noChosenEV_ratio'].mean() if blockType != 'SRO_only' else 0 if blockType != 'SRO_only' else np.nan
        log_chosenEV_noChosenEV_ratio_std = sub_block_data['log_chosenEV_noChosenEV_ratio'].std() if blockType != 'SRO_only' else 0 if blockType != 'SRO_only' else np.nan
        chosenEV_noChosenEV_diff_mean = sub_block_data['chosenEV_noChosenEV_diff'].mean() if blockType != 'SRO_only' else np.nan
        chosenEV_noChosenEV_diff_std = sub_block_data['chosenEV_noChosenEV_diff'].std() if blockType != 'SRO_only' else np.nan
        log_chosenMagnitude_noChosenMagnitude_ratio_mean = sub_block_data['log_chosenMagnitude_noChosenMagnitude_ratio'].mean() if blockType != 'SRO_only' else np.nan
        log_chosenMagnitude_noChosenMagnitude_ratio_std = sub_block_data['log_chosenMagnitude_noChosenMagnitude_ratio'].std() if blockType != 'SRO_only' else np.nan
        chosenMagnitude_noChosenMagnitude_diff_mean = sub_block_data['chosenMagnitude_noChosenMagnitude_diff'].mean() if blockType != 'SRO_only' else np.nan
        chosenMagnitude_noChosenMagnitude_diff_std = sub_block_data['chosenMagnitude_noChosenMagnitude_diff'].std() if blockType != 'SRO_only' else np.nan
        log_chosenProbability_noChosenProbability_ratio_mean = sub_block_data['log_chosenProbability_noChosenProbability_ratio'].mean() if blockType != 'SRO_only' else np.nan
        log_chosenProbability_noChosenProbability_ratio_std = sub_block_data['log_chosenProbability_noChosenProbability_ratio'].std() if blockType != 'SRO_only' else np.nan
        chosenProbability_noChosenProbability_diff_mean = sub_block_data['chosenProbability_noChosenProbability_diff'].mean() if blockType != 'SRO_only' else np.nan
        chosenProbability_noChosenProbability_diff_std = sub_block_data['chosenProbability_noChosenProbability_diff'].std() if blockType != 'SRO_only' else np.nan
        stay_switch_button = sub_block_data['stay_switch_button'].value_counts().stay if 'stay' in sub_block_data['stay_switch_button'].value_counts() else 0 if blockType != 'SRO_only' else np.nan
        stay_switch_location = sub_block_data['stay_switch_location'].value_counts().stay if 'stay' in sub_block_data['stay_switch_location'].value_counts() else 0 if blockType != 'SRO_only' else np.nan
        StayOnKeyANDLocation = sub_block_data['StayOnKeyANDLocation'].sum() if blockType != 'SRO_only' else np.nan
        # utility_diff_mean = sub_block_data['utility_diff'].mean() if blockType != 'SRO_only' else np.nan
        # utility_diff_std = sub_block_data['utility_diff'].std() if blockType != 'SRO_only' else np.nan
        # utility_diff_abs_log_ratio_mean = sub_block_data['utility_diff_abs_log_ratio'].mean() if blockType != 'SRO_only' else np.nan
        # utility_diff_abs_log_ratio_std = sub_block_data['utility_diff_abs_log_ratio'].std() if blockType != 'SRO_only' else np.nan
        chose_bottom = sub_block_data['chose_bottom'].sum() if blockType != 'SRO_only' else np.nan
        chose_safer = sub_block_data['chose_safer'].sum() if blockType != 'SRO_only' else np.nan
        chose_best_EV = sub_block_data['chose_best_EV'].sum() if blockType != 'SRO_only' else np.nan

        # SRO stuff:
        non_perfect_seq_pressing = sub_block_data['non_perfect_seq_pressing'].sum() if blockType != 'gambles_only' else np.nan
        sequenceCompleted = sub_block_data['sequenceCompleted'].sum() if blockType != 'gambles_only' else np.nan
        isWin = sub_block_data['isWin'].sum() if blockType != 'gambles_only' else np.nan
        total_cost_mean = sub_block_data['total_cost'].mean() if blockType != 'gambles_only' else np.nan
        total_cost_std = sub_block_data['total_cost'].std() if blockType != 'gambles_only' else np.nan
        SRO_rt_mean = sub_block_data['SRO_rt'].mean() if blockType != 'gambles_only' else np.nan
        SRO_rt_std = sub_block_data['SRO_rt'].std() if blockType != 'gambles_only' else np.nan
        SRO_rt_of_SRO_key_mean = sub_block_data['SRO_rt_of_SRO_key'].mean() if blockType != 'gambles_only' else np.nan
        SRO_rt_of_SRO_key_std = sub_block_data['SRO_rt_of_SRO_key'].std() if blockType != 'gambles_only' else np.nan
        SRO_seq_completion_time_mean = sub_block_data['SRO_seq_completion_time'].mean() if blockType != 'gambles_only' else np.nan
        SRO_seq_completion_time_std = sub_block_data['SRO_seq_completion_time'].std() if blockType != 'gambles_only' else np.nan
        SRO_seq_completion_time_from_start_mean = sub_block_data['SRO_seq_completion_time_from_start'].mean() if blockType != 'gambles_only' else np.nan
        SRO_seq_completion_time_from_start_std = sub_block_data['SRO_seq_completion_time_from_start'].std() if blockType != 'gambles_only' else np.nan
        n_pressing_gambleKeys_in_SRO_mean = sub_block_data['n_pressing_gambleKeys_in_SRO'].mean() if blockType != 'gambles_only' else np.nan
        n_pressing_gambleKeys_in_SRO_std = sub_block_data['n_pressing_gambleKeys_in_SRO'].std() if blockType != 'gambles_only' else np.nan
        n_pressing_SRO_Keys_in_SRO_mean = sub_block_data['n_pressing_SRO_Keys_in_SRO'].mean() if blockType != 'gambles_only' else np.nan
        n_pressing_SRO_Keys_in_SRO_std = sub_block_data['n_pressing_SRO_Keys_in_SRO'].std() if blockType != 'gambles_only' else np.nan
        IPI_abs_diff_sum_mean = sub_block_data['IPI_abs_diff_sum'].mean() if blockType != 'gambles_only' else np.nan
        IPI_abs_diff_sum_std = sub_block_data['IPI_abs_diff_sum'].std() if blockType != 'gambles_only' else np.nan

        # general stuff:
        # rho = sub_block_data['rho'].values[0]
        # b = sub_block_data['b'].values[0]
        # riskModel_accuracy = sub_block_data['riskModel_accuracy'].values[0]


        # now create a dataframe with the features:
        block_summary_features = pd.DataFrame({'sub': subj, 'block': block, 'time': time, 'blockType': blockType, 'group': group, 'choice_rt_mean': choice_rt_mean, 'choice_rt_std': choice_rt_std, 'chosen_location': chosen_location, 'chosen_direction': chosen_direction, 'non_perfect_seq_pressing': non_perfect_seq_pressing, 'sequenceCompleted': sequenceCompleted, 'isWin': isWin, 'total_cost_mean': total_cost_mean, 'total_cost_std': total_cost_std, 'n_chosen_gambles': n_chosen_gambles, 'SRO_rt_mean': SRO_rt_mean, \
        'SRO_rt_std': SRO_rt_std, 'SRO_rt_of_SRO_key_mean': SRO_rt_of_SRO_key_mean, 'SRO_rt_of_SRO_key_std': SRO_rt_of_SRO_key_std, 'SRO_seq_completion_time_mean': SRO_seq_completion_time_mean, 'SRO_seq_completion_time_std': SRO_seq_completion_time_std, 'SRO_seq_completion_time_from_start_mean': SRO_seq_completion_time_from_start_mean, 'SRO_seq_completion_time_from_start_std': SRO_seq_completion_time_from_start_std, 'n_pressing_gambleKeys_in_SRO_mean': n_pressing_gambleKeys_in_SRO_mean, 'n_pressing_gambleKeys_in_SRO_std': n_pressing_gambleKeys_in_SRO_std, \
        'n_pressing_SRO_Keys_in_SRO_mean': n_pressing_SRO_Keys_in_SRO_mean, 'n_pressing_SRO_Keys_in_SRO_std': n_pressing_SRO_Keys_in_SRO_std, 'log_chosenEV_noChosenEV_ratio_mean': log_chosenEV_noChosenEV_ratio_mean, 'log_chosenEV_noChosenEV_ratio_std': log_chosenEV_noChosenEV_ratio_std, 'chosenEV_noChosenEV_diff_mean': chosenEV_noChosenEV_diff_mean, 'chosenEV_noChosenEV_diff_std': chosenEV_noChosenEV_diff_std, 'log_chosenMagnitude_noChosenMagnitude_ratio_mean': log_chosenMagnitude_noChosenMagnitude_ratio_mean, 'log_chosenMagnitude_noChosenMagnitude_ratio_std': log_chosenMagnitude_noChosenMagnitude_ratio_std, \
        'chosenMagnitude_noChosenMagnitude_diff_mean': chosenMagnitude_noChosenMagnitude_diff_mean, 'chosenMagnitude_noChosenMagnitude_diff_std': chosenMagnitude_noChosenMagnitude_diff_std, 'log_chosenProbability_noChosenProbability_ratio_mean': log_chosenProbability_noChosenProbability_ratio_mean, 'log_chosenProbability_noChosenProbability_ratio_std': log_chosenProbability_noChosenProbability_ratio_std, 'chosenProbability_noChosenProbability_diff_mean': chosenProbability_noChosenProbability_diff_mean, 'chosenProbability_noChosenProbability_diff_std': chosenProbability_noChosenProbability_diff_std, \
        'stay_switch_button': stay_switch_button, 'stay_switch_location': stay_switch_location, 'StayOnKeyANDLocation': StayOnKeyANDLocation, 'IPI_abs_diff_sum_mean': IPI_abs_diff_sum_mean, 'IPI_abs_diff_sum_std': IPI_abs_diff_sum_std, 'chose_bottom': chose_bottom, 'chose_safer': chose_safer, 'chose_best_EV': chose_best_EV}, index=[0])
        # add features to summary_features:
        summary_features = pd.concat([summary_features, block_summary_features])

summary_features = summary_features[summary_features['time']!='after'].reset_index(drop=True)

# ---------------------------------------------------------
# Get Habit Expression data (responses after devaluation)
# ---------------------------------------------------------
deval_data = main_data_df[main_data_df['time'] == 'after']
deval_data = deval_data[deval_data['stim_condition'] == 'devalued']
deval_data['phase'] = deval_data['phase'].astype(str)
deval_data_summary = deval_data.groupby(['group', 'sub', 'phase', 'stim_condition'])['sequenceCompleted'].sum()
# deval_data_summary = deval_data_summary.unstack(level=3)
deval_data_summary = deval_data_summary.reset_index()
sequence_completed_deval_data_summary = deval_data_summary.copy() # this is true oonly if runs before the next cell

deval_data = main_data_df[main_data_df['time'] == 'after']
deval_data = deval_data[deval_data['stim_condition'] == 'devalued']
deval_data['phase'] = deval_data['phase'].astype(str)
deval_data_summary = deval_data.groupby(['group', 'sub', 'phase', 'stim_condition'])['SRO_rt_of_SRO_key'].count()
# deval_data_summary = deval_data_summary.unstack(level=3)
deval_data_summary = deval_data_summary.reset_index()
# change SRO_rt column name to at_one_response
deval_data_summary = deval_data_summary.rename(columns={'SRO_rt_of_SRO_key': 'at_least_one_response'})

# combien sequence_completed_deval_data_summary with deval_data_summary:
deval_data_summary_all = pd.merge(sequence_completed_deval_data_summary, deval_data_summary, on=['group', 'sub', 'phase', 'stim_condition'], how='outer')
# make sequenceCompleted int:
deval_data_summary_all['sequenceCompleted'] = deval_data_summary_all['sequenceCompleted'].astype(int)
deval_data_summary_all = deval_data_summary_all.pivot(index=['sub', 'group', 'stim_condition'], columns='phase', values=['sequenceCompleted', 'at_least_one_response']).reset_index()
deval_data_summary_all.columns = ['_'.join(col).strip() if col[1] else col[0] for col in deval_data_summary_all.columns]

# now create sequenceCompleted_total column:
deval_data_summary_all['sequenceCompleted_total'] = deval_data_summary_all['sequenceCompleted_test'] + deval_data_summary_all['sequenceCompleted_reacquisition']
deval_data_summary_all['at_least_one_response_total'] = deval_data_summary_all['at_least_one_response_test'] + deval_data_summary_all['at_least_one_response_reacquisition']
# reorder columns:
deval_data_summary_all = deval_data_summary_all[['sub', 'group', 'stim_condition', 'sequenceCompleted_test', 'sequenceCompleted_reacquisition', 'sequenceCompleted_total', 'at_least_one_response_test', 'at_least_one_response_reacquisition', 'at_least_one_response_total']]
deval_data_summary_all.columns = ['devalued_' + col if col not in ['sub', 'group', 'stim_condition'] else col for col in deval_data_summary_all.columns]

# add a column sequenceCompleted_test_bin:
deval_data_summary_all['devalued_sequenceCompleted_test_bin'] = deval_data_summary_all['devalued_sequenceCompleted_test'].apply(lambda x: 1 if x > 0 else 0)
deval_data_summary_all['devalued_sequenceCompleted_reacquisition_bin'] = deval_data_summary_all['devalued_sequenceCompleted_reacquisition'].apply(lambda x: 1 if x > 0 else 0)
deval_data_summary_all['devalued_sequenceCompleted_total_bin'] = deval_data_summary_all['devalued_sequenceCompleted_total'].apply(lambda x: 1 if x > 0 else 0)
deval_data_summary_all['devalued_at_least_one_response_test_bin'] = deval_data_summary_all['devalued_at_least_one_response_test'].apply(lambda x: 1 if x > 0 else 0)
deval_data_summary_all['devalued_at_least_one_response_reacquisition_bin'] = deval_data_summary_all['devalued_at_least_one_response_reacquisition'].apply(lambda x: 1 if x > 0 else 0)
deval_data_summary_all['devalued_at_least_one_response_total_bin'] = deval_data_summary_all['devalued_at_least_one_response_total'].apply(lambda x: 1 if x > 0 else 0)
deval_data_summary_all['any_DTH_slips'] = (deval_data_summary_all['devalued_at_least_one_response_total']>0) # The same as previous line techinically (but just in a way I have used before)...

# remove group and stim_condition columns:
deval_data_summary_all = deval_data_summary_all.drop(columns=['group', 'stim_condition'])
deval_data_summary_all.head()


# ---------------------------------------------------------
# Create a dictionary of pre_deval_blocks:
# ---------------------------------------------------------
all_pre_deval_blocks = summary_features[summary_features['time']!='after'].reset_index(drop=True).block.unique()
pre_deval_blocks = {'all': all_pre_deval_blocks}

In [25]:
deval_data_summary_all.shape
# save deval_data_summary_all as pickle:
deval_data_summary_all.to_pickle('parsed_data/deval_data_summary_all_SAMPLE2.pkl')

In [29]:
summary_features

,sub,block,time,blockType,group,choice_rt_mean,choice_rt_std,chosen_location,chosen_direction,non_perfect_seq_pressing,sequenceCompleted,isWin,total_cost_mean,total_cost_std,n_chosen_gambles,SRO_rt_mean,SRO_rt_std,SRO_rt_of_SRO_key_mean,SRO_rt_of_SRO_key_std,SRO_seq_completion_time_mean,SRO_seq_completion_time_std,SRO_seq_completion_time_from_start_mean,SRO_seq_completion_time_from_start_std,n_pressing_gambleKeys_in_SRO_mean,n_pressing_gambleKeys_in_SRO_std,n_pressing_SRO_Keys_in_SRO_mean,n_pressing_SRO_Keys_in_SRO_std,log_chosenEV_noChosenEV_ratio_mean,log_chosenEV_noChosenEV_ratio_std,chosenEV_noChosenEV_diff_mean,chosenEV_noChosenEV_diff_std,log_chosenMagnitude_noChosenMagnitude_ratio_mean,log_chosenMagnitude_noChosenMagnitude_ratio_std,chosenMagnitude_noChosenMagnitude_diff_mean,chosenMagnitude_noChosenMagnitude_diff_std,log_chosenProbability_noChosenProbability_ratio_mean,log_chosenProbability_noChosenProbability_ratio_std,chosenProbability_noChosenProbability_diff_mean,chosenProbability_noChosenProbability_diff_std,stay_switch_button,stay_switch_location,StayOnKeyANDLocation,IPI_abs_diff_sum_mean,IPI_abs_diff_sum_std,chose_bottom,chose_safer,chose_best_EV
0,5001,1,NaN,gambles_only,0,685.363636,215.583521,5.0,4.0,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.066636,0.746977,-0.036364,1.150237,-0.036091,0.886018,0.363636,2.907670,-0.030364,0.840128,2.523234e-18,0.421307,5.0,8.0,4.0,NaN,NaN,5.0,5.0,6.0
1,5001,2,NaN,SRO_only,0,NaN,NaN,NaN,NaN,21.0,11.0,5.0,-3.333333,2.435843,NaN,743.866667,230.358685,743.866667,230.358685,1006.000000,366.674242,1988.454545,224.293720,0.000000,0.000000,3.428571,2.461126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,211.000000,132.242202,NaN,NaN,NaN
2,5001,3,NaN,SRO_only,0,NaN,NaN,NaN,NaN,21.0,13.0,6.0,-3.238095,2.364419,NaN,506.500000,124.923207,506.500000,124.923207,1097.384615,402.734308,1941.307692,239.107153,0.000000,0.000000,3.428571,2.521338,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,460.538462,331.075423,NaN,NaN,NaN
3,5001,4,NaN,dual,0,430.483871,153.284022,16.0,14.0,33.0,22.0,14.0,-3.424242,2.437087,31.0,691.782609,246.164756,691.782609,246.164756,937.454545,370.647147,2017.318182,340.289986,0.060606,0.242306,3.515152,2.513976,-0.015323,1.006990,-0.130645,1.768930,0.113806,1.102559,0.225806,3.509128,-0.129097,1.258579,1.612903e-03,0.423573,13.0,12.0,10.0,265.000000,270.711057,16.0,15.0,14.0
4,5001,5,NaN,gambles_only,0,685.636364,284.387859,6.0,6.0,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.261091,1.090840,0.631818,1.955668,0.268545,1.077892,1.090909,3.618136,-0.007455,1.200090,-1.363636e-02,0.417188,1.0,5.0,1.0,NaN,NaN,6.0,5.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2833,5302,7,NaN,dual,0,797.650000,92.795801,5.0,7.0,11.0,22.0,17.0,-2.000000,1.436141,22.0,1017.240909,158.661487,1017.240909,158.661487,909.150000,75.795676,1926.390909,183.101504,0.000000,0.000000,2.000000,1.436141,0.470409,0.851651,0.702273,1.549768,0.058273,0.755335,0.000000,2.742956,0.412136,0.995412,1.477273e-01,0.358395,10.0,10.0,8.0,108.331818,71.176028,5.0,15.0,15.0
2834,5302,8,NaN,dual,0,802.624138,100.344516,9.0,10.0,12.0,21.0,17.0,-2.000000,1.436141,29.0,1070.836364,215.518917,1070.836364,215.518917,989.471429,160.218096,2069.233333,301.629817,0.000000,0.000000,2.030303,1.468095,0.593310,0.785857,0.786207,1.021020,-0.129103,0.934969,-0.344828,2.831908,0.722448,1.010677,2.465517e-01,0.359049,11.0,12.0,5.0,303.863636,317.270807,9.0,22.0,23.0
2835,5302,9,NaN,dual,0,796.644444,93.534865,5.0,9.0,11.0,22.0,11.0,-2.000000,1.436141,27.0,1073.559091,184.224668,1073.559091,184.224668,999.318182,214.037126,2072.877273,243.902258,0.000000,0.000000,2.000000,1.436141,0.278444,0.667143,0.272222,1.359464,-0.402333,0.725039,-1.370370,2.203985,0.680852,0.898810,2.796296e-01,0.380602,1

In [26]:
summary_features
# save summary_features as pickle:
summary_features.to_pickle('parsed_data/summary_features_SAMPLE2.pkl')